# Milestone 7 - Inside the model: how an LLM picks its words

Milestones 1-6 treat the model as a black box: text goes in, text comes out.
This one opens the box a little. You will run six small **experiments** on a
model running on your own computer, and see for yourself:

1. **Tokens** - the model does not read letters or words, it reads pieces.
2. **Chances** - for every next piece, the model scores every option.
3. **Picking** - greedy, top-k, top-p and temperature: how one option gets chosen.
4. **Speed** - why reading your prompt and writing the answer run at very different speeds, and what the cache saves.
5. **Size** - why a 12-billion-number model fits in 8 GB.
6. **Fair comparisons** - how to tell if one model is really faster than another.

**Why bother?** Each of these explains something you will run into when you
build with LLMs: why models can't count letters, why the same prompt gives
different answers, why long prompts feel slow, why keeping the start of a
prompt the same saves money, and why one impressive answer proves nothing.

**How each experiment works.** It is the scientific method, small:

- **Question** - what we want to know.
- **Your guess** - write it down *before* you run anything. A guess you wrote
  down can be wrong, which is how you learn. A guess you didn't write down is
  always "right" afterwards.
- **Run it** - one cell, one change.
- **What happened** - compare with your guess.

Then a short exercise, with tests, the same as the other milestones.

**Do you need Ollama?** It's best with it: install it, then run
`ollama pull gemma3:12b` (and `ollama pull qwen3:14b` for experiment 6). Without
it, every experiment falls back to real measurements recorded on a MacBook Pro
(M5 Pro, 24 GB), so the notebook still works - you just can't change the inputs.

In [ ]:
import json
import os
import random
import urllib.request

import ipytest
import pytest

ipytest.autoconfig(clean="test_*")

OLLAMA = "http://localhost:11434"
MODEL = "gemma3:12b"          # used for experiments 1-5
COMPARE = ["gemma3:12b", "qwen3:14b"]   # experiment 6


def ollama_is_running():
    if os.getenv("DRILL_NOTEBOOK_OFFLINE") == "1":   # the test suite sets this
        return False
    try:
        urllib.request.urlopen(OLLAMA + "/api/version", timeout=1)
        return True
    except OSError:
        return False


LIVE = ollama_is_running()
print("Live: talking to Ollama on this computer." if LIVE
      else "Offline: using recorded measurements. Start Ollama and re-run this cell to go live.")


def generate(prompt, model=MODEL, raw=False, choices=0, **options):
    """Send one prompt to Ollama and return its full reply, numbers included.

    raw=True sends your text exactly as typed, with no chat formatting added.
    choices=N also returns the model's top N options for each piece it wrote.
    options are sampling settings such as temperature, top_k, top_p, seed.
    """
    body = {"model": model, "prompt": prompt, "stream": False, "raw": raw,
            "think": False, "options": options}
    if choices:
        body.update(logprobs=True, top_logprobs=choices)
    request = urllib.request.Request(OLLAMA + "/api/generate", json.dumps(body).encode(),
                                     {"content-type": "application/json"})
    return json.load(urllib.request.urlopen(request, timeout=300))


def normalize(choices):
    """Scale chances so they add up to 1 again, keeping their proportions."""
    total = sum(choices.values())
    return {word: chance / total for word, chance in choices.items()}


def show(choices):
    """Print chances as a little bar chart, most likely first."""
    for word, chance in sorted(choices.items(), key=lambda p: p[1], reverse=True):
        print(f"{word!r:>14}  {chance:6.1%}  {'#' * round(chance * 40)}")


# Real measurements from gemma3:12b and qwen3:14b on a MacBook Pro M5 Pro (24 GB).
RECORDED = {
 "token_counts": {
  "gravity": 2,
  "What is gravity?": 5,
  "strawberry": 2,
  "antidisestablishmentarianism": 6,
  "1234567": 9,
  "🙂": 2,
  "def two_sum(nums, target):": 10
 },
 "choices": {
  "The capital of France is": {
   " Paris": 0.9541,
   " a": 0.0061,
   " known": 0.0051,
   " the": 0.0049,
   " one": 0.003,
   " **": 0.0026,
   ",": 0.0017,
   " not": 0.0014
  },
  "My favourite colour is": {
   " blue": 0.4631,
   " green": 0.1501,
   " purple": 0.0905,
   " turquoise": 0.03,
   " Blue": 0.0281,
   " a": 0.0152,
   " emerald": 0.0115,
   " teal": 0.0114
  },
  "The opposite of hot is": {
   " cold": 0.9202,
   " not": 0.0304,
   " cool": 0.0126,
   " **": 0.0081,
   ",": 0.0023,
   "...": 0.0017,
   " \"": 0.0015,
   ":": 0.0015
  },
  "Once upon a": {
   " time": 0.9997,
   " Time": 0.0001,
   " long": 0.0001,
   " a": 0.0,
   " timeless": 0.0,
   " very": 0.0,
   " day": 0.0,
   " lifetime": 0.0
  }
 },
 "samples": {
  "0": [
   "blue",
   "blue",
   "blue",
   "blue",
   "blue",
   "blue",
   "blue",
   "blue",
   "blue",
   "blue"
  ],
  "1.0": [
   "blue",
   "blue",
   "Turquoise",
   "blue",
   "blue",
   "blue",
   "blue",
   "emerald",
   "pink",
   "purple"
  ]
 },
 "timing": {
  "short": {
   "prompt_eval_count": 16,
   "prompt_eval_duration": 86057000,
   "eval_count": 33,
   "eval_duration": 973736000
  },
  "long_first": {
   "prompt_eval_count": 802,
   "prompt_eval_duration": 1346393000,
   "eval_count": 16,
   "eval_duration": 459099000
  },
  "long_repeat": {
   "prompt_eval_count": 802,
   "prompt_eval_duration": 102742000,
   "eval_count": 16,
   "eval_duration": 458632000
  }
 },
 "speed_runs": {
  "gemma3:12b": [
   33.7,
   33.5,
   33.9
  ],
  "qwen3:14b": [
   29.7,
   29.9,
   30.2
  ]
 }
}

## 1. Tokens: the model reads pieces, not letters

Before the model sees your text, a **tokenizer** chops it into **tokens**:
common words are one piece, rarer words get split into several. Each piece is
really a number (an ID) - that's all the model ever sees.

**Question.** How many tokens is each of these?

**Your guess.** Write a number next to each before running:
`gravity`, `What is gravity?`, `strawberry`, `antidisestablishmentarianism`,
`1234567`, `🙂`, `def two_sum(nums, target):`

In [ ]:
def count_tokens(text):
    if not LIVE:
        return RECORDED["token_counts"][text] - 1
    # The model also counts a hidden "start of text" marker; take it off.
    return generate(text, raw=True, num_predict=1)["prompt_eval_count"] - 1


for text in RECORDED["token_counts"]:
    print(f"{count_tokens(text):3}  {text}")

**What happened.** Common words like `gravity` and `strawberry` are one
token each. The long rare word is split into several. Numbers are split into
roughly one token per digit, and code is chopped into lots of small pieces.

**Why it matters.**

- Ask a model how many "r"s are in strawberry and it often gets it wrong. It
  never saw the letters, only one token.
- Cloud models charge per token, and limit how many fit in one request. Code
  and numbers use more tokens than plain English.
- If you have Ollama running, try your own words, or a sentence in another
  language: `count_tokens("...")`.

## 2. Chances: the model scores every possible next piece

After reading your tokens, the model gives **every** token it knows a score,
then turns the scores into **chances** that add up to 100%. That's step 5 in
most "how LLMs work" diagrams (the LM head and softmax). Ollama can show us the
top few.

**Question.** For each sentence start, how sure is the model about the next word?

**Your guess.** Rank these from most sure to least sure:
"The capital of France is", "My favourite colour is", "The opposite of hot is",
"Once upon a".

In [ ]:
def next_word_choices(prompt, how_many=8):
    if not LIVE:
        return RECORDED["choices"][prompt]
    reply = generate(prompt, raw=True, choices=how_many, num_predict=1, temperature=0)
    import math
    return {c["token"]: math.exp(c["logprob"]) for c in reply["logprobs"][0]["top_logprobs"]}


for prompt in RECORDED["choices"]:
    print(f"\n{prompt} ...")
    show(next_word_choices(prompt))

**What happened.** "Once upon a" is almost 100% " time". "Paris" and "cold"
are well above 90%. But "My favourite colour is" spreads out: blue, green,
purple, and more, because there is no single right answer.

Notice the leading spaces, like `' Paris'`: the space is part of the token.

**Why it matters.** The model *always* picks something, even when no option is
strong. A confident-sounding sentence can sit on top of a 15% chance. When a
model makes something up, this is usually where it happened.

**Your turn.** The simplest way to choose is to always take the most likely
option. That's called **greedy** picking. Write `greedy_pick(choices)`: given
a dict of word -> chance, return the word with the highest chance.

**How to start.** It's the "best so far" pattern from milestone 0, but over a
dict. Loop with `for word, chance in choices.items():` and keep track of the
best word and its chance.

In [ ]:
def greedy_pick(choices):
    """Return the word with the highest chance."""
    # TODO: keep the best word so far and its chance; return the best word.
    raise NotImplementedError("Milestone 7: implement greedy_pick")

In [ ]:
%%ipytest -qq

def test_picks_the_most_likely_word():
    assert greedy_pick({" blue": 0.46, " green": 0.15, " purple": 0.09}) == " blue"


def test_order_in_the_dict_does_not_matter():
    assert greedy_pick({" a": 0.01, " Paris": 0.95, " the": 0.02}) == " Paris"


def test_single_option():
    assert greedy_pick({" time": 1.0}) == " time"

## 3. Picking: greedy, top-k, top-p and temperature

Greedy picking gives the same answer every time, which can be dull or get
stuck repeating itself. So most apps pick **randomly, weighted by the
chances**: a 46% word gets picked about half the time. Three settings control
how adventurous that is:

- **Temperature** reshapes the chances. Low (like 0.2) makes the favourite even
  more of a favourite. High (like 1.5) evens things out. At 0 it's greedy.
- **Top-k** throws away everything except the k most likely words, then picks
  among those.
- **Top-p** keeps the most likely words until their chances add up to p (like
  0.9), and throws away the rest.

**Question.** Finish "My favourite colour is" ten times at temperature 0,
then ten times at temperature 1. How many *different* colours do you get each
time? (From experiment 2: blue had about a 46% chance, green 15%, purple 9%.)

**Your guess.** Write two numbers down.

In [ ]:
def ten_answers(temperature):
    if not LIVE:
        return RECORDED["samples"][str(temperature)]
    # A new random seed each time, so each answer is a fresh draw.
    return [
        generate("My favourite colour is", raw=True, num_predict=1,
                 temperature=temperature, seed=random.randint(0, 10**6))["response"].strip()
        for _ in range(10)
    ]


for temperature in [0, 1.0]:
    answers = ten_answers(temperature)
    print(f"temperature {temperature}: {answers}  ->  {len(set(answers))} different")

**What happened.** At temperature 0 it's "blue" every time: that's greedy
picking. At 1 it's still mostly blue, since blue has the biggest chance, but
other colours turn up in about the proportions their chances suggest.

Try the same thing with a question that has one obvious answer, like "Name one
fruit": you'll get "Apple" nearly every time even at temperature 1. Temperature
can only spread the picks out when the model already had some doubt.

Here is what temperature does to the "favourite colour" chances. Watch the
gap between blue and the rest:

In [ ]:
def with_temperature(choices, temperature):
    """Low temperature sharpens the chances, high temperature flattens them."""
    return normalize({w: c ** (1 / temperature) for w, c in choices.items()})


colours = RECORDED["choices"]["My favourite colour is"]
for temperature in [0.5, 1.0, 2.0]:
    print(f"\ntemperature {temperature}")
    show(with_temperature(colours, temperature))

**Why it matters.** Pick the setting for the job. A tutor giving hints, a
data-extraction step, or a test you want to repeat: low temperature. Brainstorming
names or writing a story: higher. Our interview tutor uses 0.2.

**Your turn.** Write the two filters. Both return the words they keep, with
their chances re-scaled to add up to 1. The `normalize` helper from the setup
cell does that re-scaling for you.

- `keep_top_k(choices, k)`: keep the k most likely words.
- `keep_top_p(choices, p)`: keep the most likely words, adding them one at a
  time, until their chances add up to at least p.

**How to start.** Both start the same way: put the words in order, most likely
first. `sorted(choices.items(), key=lambda pair: pair[1], reverse=True)` gives
you a list of `(word, chance)` pairs in that order. For top-k, take the first
k. For top-p, loop over the pairs with a running total and stop once it
reaches p.

In [ ]:
def keep_top_k(choices, k):
    """Keep the k most likely words, re-scaled so their chances add up to 1."""
    # TODO: sort most likely first, keep the first k, then normalize.
    raise NotImplementedError("Milestone 7: implement keep_top_k")


def keep_top_p(choices, p):
    """Keep the most likely words until their chances reach p, re-scaled to add up to 1."""
    # TODO: sort most likely first, add words until the running total reaches p, then normalize.
    raise NotImplementedError("Milestone 7: implement keep_top_p")

In [ ]:
%%ipytest -qq

COLOURS = {" blue": 0.5, " green": 0.3, " purple": 0.15, " teal": 0.05}


def test_top_k_keeps_the_k_most_likely():
    assert set(keep_top_k(COLOURS, 2)) == {" blue", " green"}


def test_top_k_rescales_to_add_up_to_one():
    kept = keep_top_k(COLOURS, 2)
    assert kept[" blue"] == pytest.approx(0.625)
    assert sum(kept.values()) == pytest.approx(1.0)


def test_top_p_stops_once_the_total_is_reached():
    """0.5 + 0.3 = 0.8 is enough for p=0.8, so purple is not needed."""
    assert set(keep_top_p(COLOURS, 0.8)) == {" blue", " green"}


def test_top_p_keeps_one_word_when_the_model_is_sure():
    """The point of top-p: how many words survive depends on how sure the model is."""
    assert set(keep_top_p({" time": 0.97, " long": 0.02, " day": 0.01}, 0.9)) == {" time"}


def test_top_p_rescales_to_add_up_to_one():
    assert sum(keep_top_p(COLOURS, 0.9).values()) == pytest.approx(1.0)

## 4. Speed: reading versus writing, and the cache

A model handles your request in two phases:

- **Reading (prefill).** It reads your whole prompt at once, all tokens in
  parallel. This decides how long you wait for the *first* word.
- **Writing (decode).** It writes the answer **one token at a time**, and each
  new token needs a full pass through the model. This decides how fast the
  words stream out.

While reading, the model saves its working notes for every token (the **KV
cache**). If your next prompt starts with exactly the same text, it can reuse
those notes instead of reading again.

**Question.** Compare a short prompt, a long prompt (about 800 tokens), and the
same long prompt sent a second time.

**Your guess.** Which phase gets slower with a long prompt: reading, writing,
or both? And how much faster is the repeat?

In [ ]:
def timing_runs():
    if not LIVE:
        return RECORDED["timing"]
    keys = ["prompt_eval_count", "prompt_eval_duration", "eval_count", "eval_duration"]
    # A random first line makes the long prompt new, so the first send can't use the cache.
    background = f"Note {random.randint(0, 10**9)}. " + "Gravity is the force that pulls objects with mass toward each other. " * 60
    long_prompt = background + "Now explain gravity in one sentence."
    generate("warm up", num_predict=1)   # load the model first, so loading time isn't counted
    runs = {}
    for name, prompt in [("short", "Explain gravity in one sentence."),
                         ("long_first", long_prompt), ("long_repeat", long_prompt)]:
        reply = generate(prompt, num_predict=40, temperature=0)
        runs[name] = {k: reply[k] for k in keys}
    return runs


TIMING = timing_runs()
for name, stats in TIMING.items():
    print(f"{name:12} read {stats['prompt_eval_count']:4} tokens in "
          f"{stats['prompt_eval_duration'] / 1e9:5.2f}s | wrote {stats['eval_count']:3} tokens in "
          f"{stats['eval_duration'] / 1e9:5.2f}s")

**What happened.** The long prompt took much longer to *read* - more tokens,
more reading. Writing speed hardly changed: each written token costs about the
same however long the prompt was. And the repeat was read many times faster,
because the cache already held its notes.

**Why it matters.**

- A long prompt delays the first word, but doesn't slow the stream after it.
- Put the parts of your prompt that never change (instructions, examples,
  documents) **first**, and the parts that change (the user's question)
  **last**. Then the unchanged start can be reused. Cloud providers call this
  **prompt caching** and charge much less for cached tokens.

**Your turn.** Write `speeds(stats)`, which turns one of those Ollama records
into tokens per second for each phase. Return a dict with `"reading"` and
`"writing"`.

**How to start.** Speed is count divided by seconds. Ollama gives you
`prompt_eval_count` and `prompt_eval_duration` for reading, and `eval_count`
and `eval_duration` for writing. The durations are in **nanoseconds**: divide
by `1_000_000_000` to get seconds.

In [ ]:
def speeds(stats):
    """Return {"reading": tokens per second, "writing": tokens per second}."""
    # TODO: count / seconds for each phase; durations are in nanoseconds.
    raise NotImplementedError("Milestone 7: implement speeds")

In [ ]:
%%ipytest -qq

def test_speeds_in_tokens_per_second():
    stats = {"prompt_eval_count": 800, "prompt_eval_duration": 2_000_000_000,
             "eval_count": 30, "eval_duration": 1_000_000_000}
    assert speeds(stats) == {"reading": 400.0, "writing": 30.0}


def test_reading_is_much_faster_than_writing_on_real_data():
    s = speeds(RECORDED["timing"]["long_first"])
    assert s["reading"] > 10 * s["writing"]


def test_the_cache_makes_the_repeat_read_faster():
    first = speeds(RECORDED["timing"]["long_first"])["reading"]
    repeat = speeds(RECORDED["timing"]["long_repeat"])["reading"]
    assert repeat > 5 * first

In [ ]:
for name, stats in TIMING.items():
    s = speeds(stats)
    print(f"{name:12} reading {s['reading']:8.0f} tokens/s   writing {s['writing']:5.1f} tokens/s")

## 5. Size: why a 12-billion-number model fits in 8 GB

A model is mostly a huge list of numbers called **weights**. "12B" means about
12 billion of them. How much memory that takes depends on how precisely each
number is stored, measured in **bits**:

- 16 bits per number is the usual full size.
- **Quantization** rounds the numbers so each one takes fewer bits, often 8 or 4.
  The model gets smaller and faster, and usually only a little less accurate.

**Question.** How many gigabytes do 12 billion numbers take at 16, 8 and 4 bits?

**Your guess.** Write three numbers down. Then compare with the size Ollama
reports for `gemma3:12b`: about 8 GB.

**Your turn.** Write `weights_size_gb(billions, bits)`.

**How to start.** 8 bits make 1 byte, so each number takes `bits / 8` bytes.
A billion bytes is a gigabyte. So the answer is billions times bytes-per-number.

In [ ]:
def weights_size_gb(billions, bits):
    """Gigabytes needed to store `billions` billion numbers at `bits` bits each."""
    # TODO: bytes per number is bits / 8; multiply by the number of billions.
    raise NotImplementedError("Milestone 7: implement weights_size_gb")

In [ ]:
%%ipytest -qq

def test_full_size_is_two_bytes_per_number():
    assert weights_size_gb(12, 16) == 24


def test_four_bits_is_a_quarter_of_that():
    assert weights_size_gb(12, 4) == 6


def test_your_laptop_fits_a_14b_model_at_4_bits():
    assert weights_size_gb(14, 4) < 16

In [ ]:
for bits in [16, 8, 4]:
    print(f"12B at {bits:2} bits: {weights_size_gb(12, bits):4.0f} GB")

**What happened.** At 16 bits, 24 GB - more than a 24 GB laptop can give a
model, since the operating system and your apps need room too. At 4 bits, 6 GB. Ollama's
download is about 8 GB because a few parts are kept at higher precision, and
Gemma 3 also includes an image reader.

**Why it matters.** This is the whole reason you can run models locally. It's
also why the 1-trillion-number models (like Kimi) can't run on a laptop at
all: even at 4 bits that's about 500 GB.

On top of the weights, the **KV cache** from experiment 4 also needs memory,
and it grows with every token in the conversation. That's why very long
conversations can slow down or run out of room on small machines.

## 6. Fair comparisons: is Gemma really faster than Qwen?

Everything a model does has some randomness, and your computer is busy with
other things too. So **one run proves nothing**. A fair comparison needs:

- **One change at a time.** Same prompt, same settings, only the model differs.
- **Several runs of each.** Three is a minimum, five is better.
- **A rule for the verdict, decided before you look.** Ours is blunt: only call
  a winner if *every* run of one beats *every* run of the other. If the ranges
  overlap, it's too close to call.

That rule is cautious on purpose: it never crowns a fake winner, but it can miss
a real one, because overlapping ranges don't prove there's no difference. It
suits speed runs like this one, where the runs aren't paired up. When both
setups answer **the same examples**, the standard is a **paired comparison**
(McNemar's test), which you'll meet below.

**Question.** Which writes faster on this computer, `gemma3:12b` or `qwen3:14b`?

**Your guess.** Qwen is a bit bigger (14B against 12B). Does that make it slower?

In [ ]:
def writing_speed_runs(model, runs=3):
    if not LIVE:
        return RECORDED["speed_runs"][model]
    generate("warm up", model=model, num_predict=1)
    results = []
    for _ in range(runs):
        reply = generate("Write two sentences about rivers.", model=model,
                         num_predict=80, temperature=0.7, seed=random.randint(0, 10**6))
        results.append(round(reply["eval_count"] / (reply["eval_duration"] / 1e9), 1))
    return results


SPEED = {model: writing_speed_runs(model) for model in COMPARE}
for model, runs in SPEED.items():
    print(f"{model:12} {runs} tokens/s")

**Your turn.** Write `verdict(a_runs, b_runs)`. Return `"A"` if the slowest run
of A is still faster than the fastest run of B, `"B"` for the opposite, and
`"too close to call"` otherwise.

**How to start.** `min()` and `max()` of a list give you its slowest and
fastest run. A wins when `min(a_runs) > max(b_runs)`.

In [ ]:
def verdict(a_runs, b_runs):
    """Return "A", "B" or "too close to call", using non-overlapping ranges."""
    # TODO: A wins if its worst run beats B's best run; check B the same way; else too close.
    raise NotImplementedError("Milestone 7: implement verdict")

In [ ]:
%%ipytest -qq

def test_clear_winner_a():
    assert verdict([33.7, 33.5, 33.9], [29.7, 29.9, 30.2]) == "A"


def test_clear_winner_b():
    assert verdict([20, 21], [25, 26]) == "B"


def test_overlapping_ranges_are_too_close_to_call():
    """One lucky run of A is not enough to call it."""
    assert verdict([30, 35, 31], [32, 33, 32]) == "too close to call"

In [ ]:
a, b = COMPARE
result = verdict(SPEED[a], SPEED[b])
print({"A": f"{a} is faster", "B": f"{b} is faster"}.get(result, "too close to call"))

**What happened.** On the recorded Mac, Gemma wrote about 34 tokens a second
and Qwen about 30, and the runs were tight enough to call it. Your numbers
may differ - which is the point of measuring instead of assuming.

**Speed is the easy thing to measure.** The harder, more useful question is
*quality*: does Qwen give better hints than Gemma? The method is the same:

1. Write the plan first: the question, the one change, the eval set, the
   metric, the comparison and the decision rule. `docs/EXPERIMENT_PLAN.md` has a
   template, with this exact experiment filled in as the worked example.
2. Collect wrong solutions to different problems, and check you have enough:
   with n cases, a pass rate is only good to about ±100 ÷ √n points.
3. Send the same ones to both models, same prompt, same settings, 3 times each.
4. Score every hint against a checklist you wrote first: does it give away the
   answer? does it point at the real bug?
5. Compare **case by case**. Cases where both pass or both fail tell you
   nothing; count only the disagreements. Call a winner only if the gap between
   the two counts is at least 2 × √(disagreements). That's McNemar's test in
   plain words, and the Daily Drill lesson "Comparing on the same examples"
   walks through it.

MLflow can record every run and score hints automatically with a "judge"
model (see `docs/CURRICULUM.md`, *Using MLflow on yourself*). One warning:
don't let a model judge its own answers.

## 7. What this notebook can't show you

Ollama is built to run models fast, not to open them up, so three parts of the
usual diagram stayed hidden:

- **Embeddings** - turning each token ID into a long list of numbers that
  captures its meaning.
- **The transformer layers and attention** - where each token "looks at" the
  earlier tokens to decide what matters. This is most of the model.
- **Speculative decoding** - a small model guesses a few tokens ahead and the
  big model checks them all at once, so the same answer comes faster.

To see those, you load a small model directly in Python (with the
`transformers` library) and copy out the numbers at each layer. That's a good
next step once this one makes sense. For a picture-first tour, the interactive
3D model at <https://bbycroft.net/llm> walks through every step.

**What to take away from milestone 7.**

| You saw | So when building, you |
|---|---|
| Tokens are pieces | Expect trouble with spelling, counting and exact numbers |
| Every token has a chance | Treat confident wording as no proof of a confident model |
| Temperature and top-k/top-p | Choose low for repeatable jobs, higher for creative ones |
| Reading vs writing, the cache | Put fixed instructions first, the changing question last |
| Quantization | Know what fits on your machine, and why |
| Fair comparisons | Change one thing, run several times, decide the rule first |